# EN4554 Assignment 2: Convolutional Neural Networks

**Name:** TODO  
**Index number:** TODO  
**GitHub profile:** TODO  
**Repository:** TODO

This notebook contains the experiment setup and a scaffold for Tasks 1?4. Task implementations, measured results, and discussions are still to be completed.

**AI assistance acknowledgment:** OpenAI Codex assisted with the initial notebook structure and setup code. Update this statement to describe all assistance used in the final submission. I am responsible for understanding and checking the submitted work.

## How to use this notebook

1. In Colab, select a T4 GPU runtime; locally, use a Python environment with PyTorch, torchvision, NumPy, and Matplotlib installed.
2. Run the setup cells. The data preparation cell downloads CIFAR-10 on its first run.
3. Complete the task sections in order and record the settings used for every experiment.
4. Add interpretations beside all plots and tables. Do not use the test set before the final evaluation in optional Task 4.
5. Save outputs, commit progress regularly, and export the completed notebook as `your_index_a02.pdf`.

No training or dataset download has been performed in this starter notebook.

In [ ]:
# If dependencies are missing, install them in your environment first.
# Colab normally includes these packages.
from pathlib import Path
import json
import random
import time

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 42
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BATCH_SIZE = 128
NUM_WORKERS = 0  # Portable default for Windows notebooks and Colab.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()
print(f"PyTorch: {torch.__version__} | Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0))

## Shared data setup

Use a fixed 45,000/5,000 train/validation split. Compute normalization statistics from the 45,000 training images only, before augmentation. All tasks reuse these split indices. Task 3(a) uses a fixed 5,000-image subset of the training partition.

The test dataset is deliberately not constructed here.

In [ ]:
raw_train = datasets.CIFAR10(root=DATA_DIR, train=True, download=True)

split_generator = torch.Generator().manual_seed(SEED)
indices = torch.randperm(len(raw_train), generator=split_generator)
train_indices = indices[:45000].tolist()
val_indices = indices[45000:].tolist()
small_train_indices = train_indices[:5000]

# Accumulate pixel moments in chunks to avoid copying the full training array.
channel_sum = np.zeros(3, dtype=np.float64)
channel_square_sum = np.zeros(3, dtype=np.float64)
pixel_count = 0
for start in range(0, len(train_indices), 1000):
    pixels = raw_train.data[train_indices[start:start + 1000]].astype(np.float64) / 255.0
    channel_sum += pixels.sum(axis=(0, 1, 2))
    channel_square_sum += np.square(pixels).sum(axis=(0, 1, 2))
    pixel_count += pixels.shape[0] * pixels.shape[1] * pixels.shape[2]
mean = channel_sum / pixel_count
std = np.sqrt(np.maximum(channel_square_sum / pixel_count - mean ** 2, 0))
del pixels

normalize = transforms.Normalize(mean.tolist(), std.tolist())
eval_transform = transforms.Compose([transforms.ToTensor(), normalize])
eval_train_dataset = datasets.CIFAR10(
    root=DATA_DIR, train=True, download=False, transform=eval_transform
)
train_dataset = Subset(eval_train_dataset, train_indices)
val_dataset = Subset(eval_train_dataset, val_indices)
small_train_dataset = Subset(eval_train_dataset, small_train_indices)

split_info = {
    "seed": SEED, "train_indices": train_indices,
    "val_indices": val_indices, "small_train_indices": small_train_indices,
    "mean": mean.tolist(), "std": std.tolist(),
}
(OUTPUT_DIR / "data_split.json").write_text(json.dumps(split_info), encoding="utf-8")
print(f"Train: {len(train_dataset)} | Validation: {len(val_dataset)}")
print("Training mean:", mean)
print("Training std:", std)

In [ ]:
def make_loader(dataset, shuffle=False, seed=SEED):
    return DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=shuffle,
        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
        generator=torch.Generator().manual_seed(seed),
    )

train_loader = make_loader(train_dataset, shuffle=True)
val_loader = make_loader(val_dataset)
small_train_loader = make_loader(small_train_dataset, shuffle=True)

# Inspect training images only.
images, labels = next(iter(make_loader(train_dataset)))
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, image, label in zip(axes.flat, images[:10], labels[:10]):
    display_image = image.permute(1, 2, 0).numpy() * std + mean
    ax.imshow(np.clip(display_image, 0, 1))
    ax.set_title(raw_train.classes[label.item()])
    ax.axis("off")
plt.tight_layout()
plt.show()

## Shared training helpers

These helpers use autograd and ordinary cross-entropy for Tasks 2 onward. Task 1 requires a separate manual backward/training implementation. Mixup in Task 3(a) also requires adapting the training loss below.

For fair comparisons, reset the seed, recreate the model and optimizer, and recreate the shuffled loader for each run. Keep the epoch budget fixed within each comparison. Measure final training accuracy on unaugmented data, including for mixup experiments.

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, correct, count = 0.0, 0, 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        logits = model(inputs)
        total_loss += nn.functional.cross_entropy(logits, targets, reduction="sum").item()
        correct += (logits.argmax(dim=1) == targets).sum().item()
        count += targets.size(0)
    return {"loss": total_loss / count, "accuracy": correct / count}


def train_epoch(model, loader, optimizer):
    model.train()
    total_loss, count = 0.0, 0
    for inputs, targets in loader:
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = nn.functional.cross_entropy(model(inputs), targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * targets.size(0)
        count += targets.size(0)
    return total_loss / count


def fit(model, optimizer, train_loader, val_loader, epochs):
    model.to(DEVICE)
    history = []
    for epoch in range(1, epochs + 1):
        start = time.perf_counter()
        train_loss = train_epoch(model, train_loader, optimizer)
        validation = evaluate(model, val_loader)
        row = {"epoch": epoch, "train_loss": train_loss,
               "val_loss": validation["loss"], "val_accuracy": validation["accuracy"],
               "seconds": time.perf_counter() - start}
        history.append(row)
        print(f"Epoch {epoch:02d} | loss {train_loss:.4f} | "
              f"val accuracy {validation['accuracy']:.2%}")
    return history


def plot_history(history, title):
    epochs = [row["epoch"] for row in history]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].plot(epochs, [r["train_loss"] for r in history], label="Train")
    axes[0].plot(epochs, [r["val_loss"] for r in history], label="Validation")
    axes[0].set(xlabel="Epoch", ylabel="Cross-entropy loss")
    axes[0].legend()
    axes[1].plot(epochs, [100 * r["val_accuracy"] for r in history])
    axes[1].set(xlabel="Epoch", ylabel="Validation accuracy (%)")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def save_results(name, config, history):
    payload = {"config": config, "history": history,
               "torch_version": torch.__version__, "seed": SEED}
    (OUTPUT_DIR / f"{name}.json").write_text(
        json.dumps(payload, indent=2), encoding="utf-8"
    )

## Task 1 ? Backpropagation

### 1(a) Manual MLP

Implement `Linear`, `ReLU`, and numerically stable softmax cross-entropy with explicit forward/backward methods. Cache the inputs needed for the backward pass. Assemble an MLP accepting flattened CIFAR-10 images (3,072 inputs, 10 output logits). Do not use autograd for the manual implementation.

In [ ]:
# TODO: Implement manual Linear, ReLU, softmax cross-entropy, and MLP.
# For example: 3072 -> 256 -> 10. Record the architecture you choose.

### 1(b) Gradient verification

Use a small deterministic example and float64 precision. Compare every parameter gradient and the input gradient with autograd and centred finite differences:

`df/dx_i ? [f(x + h e_i) - f(x - h e_i)] / (2h)`.

Report relative errors, your definition of relative error, the finite-difference step size, and `torch.allclose` results with explicit tolerances. Use autograd only as the reference check.

In [ ]:
# TODO: Autograd reference, centred finite differences, and comparison table.

### 1(c) Train the manual MLP

Train with manual SGD updates on flattened CIFAR-10 images. Plot loss and report validation accuracy. Record learning rate, batch size, epoch count, and initialization.

In [ ]:
# TODO: Manual training loop, loss plot, and validation accuracy.

### 1(d) Gradient flow in a 10-layer MLP

Define clearly how you count the 10 layers. At initialization, run forward/backward on a fixed batch for sigmoid and ReLU networks. Plot per-layer gradient norms and record the initialization used so that the comparison is interpretable.

In [ ]:
# TODO: Sigmoid/ReLU gradient-norm comparison at initialization.

**Task 1 interpretation ? TODO:** Explain the gradient checks, training behavior, and how/why gradient magnitudes change with depth.

## Task 2 ? Optimizers

### 2(a) Custom momentum SGD and Adam

Implement both as `torch.optim.Optimizer` subclasses. Include Adam bias correction. Compare several consecutive updates against `torch.optim.SGD` and `torch.optim.Adam`, starting from identical parameters and using identical gradients and hyperparameters. Report numerical errors and matching checks.

In [ ]:
# TODO: MomentumSGD, CustomAdam, and multi-step matching checks.

### 2(b) Optimizer trajectories

Plot contours of `f(x, y) = 0.5 * (x**2 + 50*y**2)` and the trajectories of plain SGD, momentum SGD, and Adam. Its gradient is `(x, 50*y)`. Record the common starting point, learning rates, and iteration count.

In [ ]:
# TODO: Quadratic objective, optimizer trajectories, contour plot.

### 2(c) Learning-rate sensitivity on a small CNN

Use one fixed CNN and three learning rates for each of plain SGD, momentum SGD, and Adam (nine runs). Select each optimizer's best rate using validation results. Plot validation accuracy versus learning rate and training loss at the best rate. Record the selection rule and use a consistent training budget.

In [ ]:
# TODO: SmallCNN, nine-run experiment, result table, and plots.
# Choose learning-rate ranges appropriate to each optimizer and document them.

**Task 2 interpretation ? TODO:** Explain trajectory differences, optimizer sensitivity, and what your measured CNN results support.

## Task 3 ? Regularization and residual connections

### 3(a) Regularization on 5,000 training images

Use the fixed `small_train_indices` and a small CNN with batch normalization. Compare baseline, weight decay, dropout, flip+crop, cutout, mixup, and all combined. Implement cutout and mixup yourself.

Use separate dataset instances for training augmentation and clean accuracy measurement. Keep the validation set unchanged. Report clean training accuracy, validation accuracy, and their gap (in percentage points); plot training and validation curves. Document augmentation order and all hyperparameters.

In [ ]:
# TODO: Cutout, mixup, configurable BN CNN, and seven experiments.
# Evaluate final clean training accuracy with:
# clean_small_loader = make_loader(small_train_dataset)
# train_metrics = evaluate(model, clean_small_loader)

**Task 3(a) interpretation ? TODO:** Discuss overfitting, the accuracy gap, and whether combining regularizers improves your measured results.

### 3(b) Plain versus residual networks

Implement depths `6n + 2` = 8, 20, 56 (`n` = 1, 3, 9), using three stages with channels 16/32/64, batch normalization, Kaiming initialization, and global average pooling. Within each matched pair, the only architectural difference must be the shortcut. Document handling of shortcuts when spatial size or channel count changes.

Train all six models with a consistent protocol and plot training loss versus epoch.

In [ ]:
# TODO: Shared plain/residual implementation, six runs, training-loss plots.

**Task 3(b) interpretation ? TODO:** Explain the measured effect of depth and shortcuts on optimization. Distinguish training difficulty from overfitting.

## Task 4 ? Final CNN (optional)

Design a CNN with at most **500,000 parameters** and at most **20 training epochs**. Tune using validation data only. Report ablations covering at least three design choices, with controlled comparisons.

Before evaluating the test set, freeze the architecture, hyperparameters, checkpoint, and selection rule. Evaluate the final model on the test set once. Leave the following cell unexecuted until then.

In [ ]:
# TODO: Final model, parameter-count check, validation tuning, and ablations.
# parameter_count = sum(p.numel() for p in final_model.parameters())
# assert parameter_count <= 500_000
# assert final_epochs <= 20

In [ ]:
# FINAL TEST EVALUATION ? intentionally disabled.
# Run only once after all validation-based decisions are complete.
# final_model.load_state_dict(torch.load(final_checkpoint, map_location=DEVICE))
# final_model.to(DEVICE)
# test_dataset = datasets.CIFAR10(
#     root=DATA_DIR, train=False, download=True, transform=eval_transform
# )
# test_metrics = evaluate(final_model, make_loader(test_dataset))
# print(f"Final test accuracy: {test_metrics['accuracy']:.2%}")

**Task 4 interpretation ? TODO:** Explain the design choices, ablation results, parameter budget, and final measured test accuracy.

## Submission checklist

- [ ] Name, index number, and GitHub profile included.
- [ ] Regular commits show progress over time.
- [ ] Tasks 1?3 complete; Task 4 clearly identified as optional.
- [ ] Experimental settings and measured results recorded.
- [ ] Important code, legible plots, comparison tables, and interpretations included.
- [ ] AI assistance acknowledgment updated accurately.
- [ ] Completed notebook checked for reproducibility.
- [ ] PDF export checked for clipped code, missing figures, and unreadable tables.
- [ ] PDF named `your_index_a02.pdf` (replace `your_index` with your index number).

The assignment states that only the PDF is used for grading; do not rely on external files to communicate essential results.